# Lab 12, Capstone: A Reproducible Distillation Study

**Tier 2 lab.** Part A executes and asserts anywhere, because it *is* the study's
scaffolding; Part B is the study itself, run when `RUN_STUDY = True`.

**The question is yours.** The course's suggested default, chosen because every tool it needs
already exists in your `runs/` directory: **off-policy cached-logit distillation versus
on-policy distillation at matched total compute, across two student sizes.** Off-policy here
means training on precomputed teacher logits over a fixed corpus; on-policy means the student
generates its own text during training and the teacher scores it. It is a real open
trade-off: the on-policy literature claims sample-efficiency wins, meaning more learning per
example; the off-policy pipeline claims throughput wins, meaning more examples per unit of
compute; and "matched compute" is exactly where those two claims collide, so it is the fair
place to test them. Both arms are pipelines you have already run and instrumented.

**What "capstone" means here is not "bigger". It is "auditable".** The difference between a
practitioner's experiment and an SME's study is that a competent stranger could rerun the
latter from its artifacts alone and get the same conclusion. Concretely, that stranger needs
five things, and each is a Part A section with assertions:

1. A **pre-registered protocol**. Pre-registration means writing the full plan down
   (hypotheses, arms, seeds, metrics, stopping and exclusion rules) and freezing it, by
   hashing it, before any run starts, so nobody, including you, can quietly adjust the plan
   after seeing results.
2. A **compute-matching rule** that survives scrutiny, since "matched" is where studies
   quietly cheat.
3. A **power sanity check**: is the planned number of seeds even capable of resolving the
   effect you claim to test? Power is an experiment's ability to detect an effect of a given
   size above its own noise. Lab 05's measured seed spread is the input.
4. A **manifest chain**: every artifact hash-linked to its inputs, so the provenance graph
   (the record of what produced what, all the way back) is checkable mechanically.
5. A **report skeleton** whose tables are generated from the manifests, not typed, so the
   report cannot silently disagree with the artifacts.

This notebook is deliberately the *thinnest* in the course: nearly everything it does is
calling machinery from Labs 03–11. That is the point. By Lab 12, the pipeline is
vocabulary.

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # widget progress bars crash some notebook stacks; plain logs are fine

import sys, os, json, math, hashlib, glob, time
sys.path.insert(0, "../code")

import torch

from kd_pipeline import set_seed_everywhere, config_fingerprint, RunManifest

RUN_STUDY = False        # <-- flip on the training box
SEED0 = 1000             # study seeds are SEED0 + i, disjoint from every course lab
set_seed_everywhere(SEED0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device} | RUN_STUDY: {RUN_STUDY}")

torch 2.13.0+cpu | device: cpu | RUN_STUDY: False


## Part A · 1: Pre-register the protocol, then freeze it

Everything a referee would ask for, written *before* results exist, then hashed. The hash is
the commitment: the report must cite it, and any post-hoc protocol change is visible as a
hash change, because changing even one character of the frozen file changes its hash. (Lab 05
registered predictions; this registers the entire study. Same discipline, grown up.)

Note the two easily-forgotten clauses that make protocols survivable. First, an **exclusion
rule**: a decision, made in advance, about what happens when a run collapses, and Lab 07 says
some will. Deciding this before results exist matters because deciding it afterward lets you
exclude exactly the runs that hurt your preferred conclusion. Second, a **primary metric**
singled out in advance, with everything else demoted to secondary status. The primary metric
is the one the study's verdict hangs on; secondary metrics are reported but cannot decide the
hypotheses. The reason for the split: a study that treats ten metrics as co-primary can
always find one that moved by chance, and then tell the story around it.

In [2]:
PROTOCOL = {
  "title": "Off-policy cached-logit vs on-policy distillation at matched compute",
  "hypotheses": {
    "H1": "At matched total compute, on-policy achieves higher student-rollout quality "
          "(teacher-scored) than off-policy.",
    "H2": "The on-policy advantage is larger for the smaller student "
          "(exposure bias hits weaker models harder).",
    "H0": "No difference exceeding the seed-noise band on the primary metric.",
  },
  "arms": {
    "offpolicy-360M": {"recipe": "lab04-cached-topk64", "student": "SmolLM2-360M"},
    "onpolicy-360M":  {"recipe": "lab07-gkd-lmbda1-warmstart", "student": "SmolLM2-360M"},
    "offpolicy-135M": {"recipe": "lab04-cached-topk64", "student": "SmolLM2-135M"},
    "onpolicy-135M":  {"recipe": "lab07-gkd-lmbda1-warmstart", "student": "SmolLM2-135M"},
  },
  "teacher": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
  "seeds": [1000, 1001, 1002],
  "primary_metric": "teacher-scored mean logprob of student rollouts on 128 held-out prompts",
  "secondary_metrics": ["top1 agreement (teacher-forced)", "mean entropy", "distinct-3",
                        "ECE", "benchmark subset (lab11 harness, decontaminated eval)"],
  "compute_matching": "see A2: matched by total accelerator token-passes, verified in logs",
  "stopping_rule": "fixed token-pass budget per arm; no early stopping on metrics",
  "exclusion_rule": "runs tripping the entropy monitor are kept, reported, and marked; "
                    "a config is excluded only if >=2/3 seeds trip (then reported as fragile)",
  "analysis": "per (arm, size): mean +/- range over seeds; effect claimed only if "
              "arm-mean gap > max within-arm seed range (lab05 rule)",
}
frozen = json.dumps(PROTOCOL, sort_keys=True, indent=2)
PROTO_HASH = hashlib.sha256(frozen.encode()).hexdigest()[:16]
os.makedirs("../runs/lab12", exist_ok=True)
open(f"../runs/lab12/protocol_{PROTO_HASH}.json", "w").write(frozen)

reloaded = open(f"../runs/lab12/protocol_{PROTO_HASH}.json").read()
assert hashlib.sha256(reloaded.encode()).hexdigest()[:16] == PROTO_HASH
assert len(PROTOCOL["seeds"]) >= 3 and "primary_metric" in PROTOCOL
assert "exclusion_rule" in PROTOCOL and "stopping_rule" in PROTOCOL
print(f"protocol frozen: ../runs/lab12/protocol_{PROTO_HASH}.json")
print(f"commitment hash: {PROTO_HASH}  <- cite this in the report")

protocol frozen: ../runs/lab12/protocol_c9011fd44e8a3864.json
commitment hash: c9011fd44e8a3864  <- cite this in the report


## Part A · 2: The compute-matching rule, made falsifiable

"Matched compute" fails audits when it silently means "matched steps", because on-policy
steps cost more: each one pays for generation, teacher scoring, and the student's backward
pass, while an off-policy step pays for a cached-batch backward pass alone. So matching step
counts would quietly hand the on-policy arm more total compute. The honest common currency at
fixed model sizes is **token-passes through each network**. One token-pass is one token
processed once through one network, so a forward pass over a 384-token sequence is 384
token-passes, and it works as a currency because every kind of work each recipe does
(generating, scoring, training) can be counted in it. It is priced per step below and
*verified from run logs afterward*, and that after-the-fact verification is the assertion
that makes the rule falsifiable rather than aspirational.

The table prices one optimizer step of each recipe (per unit batch × sequence). Off-policy
pays 1 student forward plus the backward, roughly 2 passes total; the teacher costs nothing
per step because its work was prepaid into the cache, whose one-time cost amortizes to
near-zero per step and is charged to the budget explicitly rather than hidden. On-policy pays
student generation (roughly 1 forward over the rollout), plus teacher scoring (1 forward),
plus student training (roughly 2), for roughly 4 passes total. So at matched budget,
off-policy takes about 2× the optimizer steps, because each of its steps costs about half as
much. That asymmetry *is the experiment*: more cheap steps versus fewer informative
ones.

In [3]:
def step_cost_token_passes(recipe, seq_len=384, rollout_len=128):
    '''Token-passes through a network per optimizer step, per batch row.'''
    if recipe.startswith("lab04"):
        return {"student": 2 * seq_len, "teacher": 0, "note": "cache prepaid"}
    if recipe.startswith("lab07"):
        return {"student": 1 * rollout_len + 2 * (seq_len + rollout_len),
                "teacher": 1 * (seq_len + rollout_len), "note": "gen + score + train"}
    raise ValueError(recipe)

BUDGET_TOKEN_PASSES = 3.0e9      # per arm, per seed; teacher prefill for the cache is
CACHE_PREPAY = 4096 * 384        # charged against the off-policy arms explicitly:

for arm, spec in PROTOCOL["arms"].items():
    c = step_cost_token_passes(spec["recipe"])
    per_step = 8 * (c["student"] + c["teacher"])          # batch 8
    prepay = CACHE_PREPAY if c["teacher"] == 0 else 0
    steps = int((BUDGET_TOKEN_PASSES - prepay) / per_step)
    print(f"{arm:<16} {c['note']:<16} {per_step:>8,} passes/step -> {steps:>7,} steps")
    assert steps > 1000, "budget must allow a meaningful run for every arm"

off = step_cost_token_passes("lab04-cached-topk64")
on  = step_cost_token_passes("lab07-gkd-lmbda1-warmstart")
ratio = (8*(on['student']+on['teacher'])) / (8*off['student'])
assert 1.5 < ratio < 4.0, "on-policy steps should cost ~2-4x off-policy steps"
print(f"\non/off cost ratio {ratio:.1f}x -> off-policy runs ~{ratio:.1f}x more steps. "
      f"That asymmetry IS the experiment.")

offpolicy-360M   cache prepaid       6,144 passes/step -> 488,025 steps
onpolicy-360M    gen + score + train   13,312 passes/step -> 225,360 steps
offpolicy-135M   cache prepaid       6,144 passes/step -> 488,025 steps
onpolicy-135M    gen + score + train   13,312 passes/step -> 225,360 steps

on/off cost ratio 2.2x -> off-policy runs ~2.2x more steps. That asymmetry IS the experiment.


## Part A · 3: Power sanity from Lab 05's measured noise

Three seeds per cell: enough? The only honest answer comes from *your* measured seed
variance, meaning the spread in results caused by nothing except changing the random seed.
Lab 05 ran every arm twice for exactly this moment. The rule, registered in the protocol:
claim an effect only if the between-arm gap exceeds the maximum within-arm seed range,
because a gap smaller than what seeds alone produce cannot be distinguished from seed noise.
This cell computes the minimum detectable effect (MDE) under that rule: the smallest true
effect the rule is able to certify, which under this rule is simply the seed spread itself.
It takes the spread from Lab 05's results if they exist, else from the course's prior
estimate, and asserts that the study's *claimed-interesting* effect size clears the MDE. If
this assertion ever fails, the fix is more seeds, not softer claims, because more seeds
shrink the noise floor while softer claims just lower the bar. Better to learn that now than
after four training runs.

In [4]:
try:
    rows = json.load(open("../runs/lab05/results.json"))
    by_arm = {}
    for r in rows:
        by_arm.setdefault(r["beta"], []).append(r["agreement"])
    seed_ranges = [max(v) - min(v) for v in by_arm.values() if len(v) > 1]
    spread, src = max(seed_ranges), "lab05 measured"
except FileNotFoundError:
    spread, src = 0.015, "course prior (run Lab 05 to replace me)"

MDE = spread                       # the registered decision rule's detection floor
INTERESTING = 0.03                 # the smallest effect H1 would care about (3 pts agreement)
print(f"max within-arm seed range: {spread:.3f} ({src})")
print(f"minimum detectable effect under the registered rule: {MDE:.3f}")
print(f"smallest effect the study calls interesting: {INTERESTING:.3f}")
assert INTERESTING > MDE, (
    "the study cannot resolve its own question at this seed count — add seeds now, "
    "not after the runs")
print("power sanity passed: 3 seeds can resolve a 3-point effect on this pair")

max within-arm seed range: 0.015 (course prior (run Lab 05 to replace me))
minimum detectable effect under the registered rule: 0.015
smallest effect the study calls interesting: 0.030
power sanity passed: 3 seeds can resolve a 3-point effect on this pair


## Part A · 4: The manifest chain, verified mechanically

Every run in this course wrote a `RunManifest`: a small JSON record naming the run's config,
seed, and its input and output artifacts by fingerprint, where a fingerprint is a hash that
changes if any of those inputs change. Chained together, these records are the study's
provenance: the checkable trail from every artifact back to whatever produced it. The
capstone's provenance requirement: **every artifact in the study's dependency graph must be
reachable through manifests, and every fingerprint must re-verify**, meaning that recomputing
the hash today gives the same value the manifest recorded. The walker below does that: it
loads every manifest under `runs/`, re-hashes what it can reach, and reports the graph. Run
here, it verifies whatever the container has (the Part A artifacts of Labs 03–11); run on the
training box, it verifies the full study. Same code in both places, which is the point: the
audit is not a special occasion, it is a function call.

In [5]:
def walk_manifests(root="../runs"):
    graph, problems = {}, []
    for path in glob.glob(f"{root}/**/manifest_*.json", recursive=True):
        m = json.load(open(path))
        graph[m["fingerprint"]] = {
            "name": m["name"], "path": path,
            "in": m.get("artifacts_in", {}), "out": m.get("artifacts_out", {})}
        recomputed = config_fingerprint({**m["config"], "seed": m["seed"]})
        if recomputed != m["fingerprint"]:
            problems.append((path, "fingerprint mismatch"))
        for tag, artifact in m.get("artifacts_out", {}).items():
            if isinstance(artifact, str) and artifact.startswith("../") \
               and not os.path.exists(artifact):
                problems.append((path, f"missing output: {artifact}"))
    return graph, problems

graph, problems = walk_manifests()
print(f"manifests found: {len(graph)} | problems: {len(problems)}")
for fp, node in list(graph.items())[:6]:
    print(f"  {fp}  {node['name']:<22} in:{len(node['in'])} out:{len(node['out'])}")
for p in problems:
    print("  PROBLEM:", p)
assert not problems, "provenance must verify before (and after) the study"
print("manifest chain verifies — the audit is a function call, not an occasion")

manifests found: 0 | problems: 0
manifest chain verifies — the audit is a function call, not an occasion


## Part B: Run the study, assemble the report

Twelve runs (4 arms × 3 seeds is 12), every one a pipeline from Labs 04 and 07 pointed at the
protocol's budget, warm-started per the protocol, babysat by Lab 07's monitor, scored by Lab
11's harness on the decontaminated eval, and manifest-chained as it lands. The report
generator then builds the tables *from the manifests*, which means the report cannot disagree
with the artifacts, because every number in it is computed from them rather than typed.

```
for seed in protocol.seeds:
    for arm, spec in protocol.arms.items():
        run = dispatch(spec.recipe, seed=seed, budget=BUDGET_TOKEN_PASSES)   # labs 04/07
        score(run, lab11_harness, decontaminated_eval)                       # lab 11
        RunManifest(..., artifacts_in={"protocol": PROTO_HASH}).save(...)
report = render(walk_manifests("../runs/lab12"), protocol)                   # tables from graph
```

The one genuinely new decision in Part B is what to do when reality diverges from the
protocol: a run collapses, a budget is breached by a retry, a seed produces an outlier. The
protocol's exclusion rule already answers each of these, because that is what it was written
in advance to do; Part B's job is to *obey it and write down that it was obeyed*. The
stranger auditing you is checking exactly this.

In [6]:
if RUN_STUDY:
    protocol = json.load(open(f"../runs/lab12/protocol_{PROTO_HASH}.json"))
    print("dispatching", len(protocol["arms"]) * len(protocol["seeds"]), "runs "
          "via labs 04/07 pipelines — see the sketch above; each run manifests itself.")
    # ... dispatch loop as sketched; then:
    # report_rows = collect from walk_manifests("../runs/lab12")
    # write ../runs/lab12/REPORT.md with: protocol hash, per-cell mean+range,
    # decision per hypothesis under the registered rule, exclusions log, limitations.
else:
    print("RUN_STUDY=False — Part B compiled but did not execute.")
    print("Twelve runs, each a lab you have already operated. The study is scheduling.")

RUN_STUDY=False — Part B compiled but did not execute.
Twelve runs, each a lab you have already operated. The study is scheduling.


## Part C: The audit, and what graduating means

**The report's required sections**, each with its generator: *Protocol* (the frozen JSON,
cited by hash); *Results* (per-cell mean ± seed range, from manifests); *Decisions* (each
hypothesis: supported / refuted / unresolved **under the registered rule**, with no narrative
promotions, meaning no upgrading a result in prose beyond what the rule certifies);
*Exclusions and deviations* (every one, with the protocol clause that governed it);
*Limitations* (the three this design cannot escape: one model family, one corpus domain, and
the fact that compute-matching by token-passes counts arithmetic while ignoring that the two
recipes stress the hardware differently, for example in how fast weights and activations can
be moved to the chip, so equal token-passes need not mean equal wall-clock; say so);
*Reproduction* (exact commands, seeds, hashes; the stranger's runbook).

**The audit checklist.** Have someone else, or you-in-two-weeks, attempt each line against
the artifacts alone:

- [ ] Protocol hash in report matches the frozen file's hash
- [ ] Every reported number regenerates from `walk_manifests` output
- [ ] Every run's manifest chains back to the protocol hash
- [ ] The primary-metric decision follows mechanically from the registered rule
- [ ] Exclusions log is complete (cross-check: 12 dispatched = reported + excluded)
- [ ] A dry-run of the reproduction commands resolves every path and hash

**Expected shape of the findings**, so you know when to be suspicious rather than excited.
H1 supported modestly for rollout quality, with teacher-forced agreement showing little or no
gap; Lab 07 taught why those two measures disagree, since one scores the student on its own
generations and the other scores it on the teacher's text. H2 is the genuinely open one, and
either outcome is interesting, which is what makes it a good capstone question. A *huge*
on-policy win on every metric more likely means the compute matching leaked, meaning one arm
quietly got more compute than the rule intended; audit the logged token-passes first.

**What graduating means.** Look at what you just did without noticing: you specified a
falsifiable question, priced it, powered it, ran it on infrastructure you built and verified
yourself, judged it with an eval you decontaminated yourself, and produced a study a stranger
can audit. That is the working definition of subject-matter expertise this course opened
with. The remaining distance to the research frontier is reading current papers, which you
now do with an operator's eye for what their protocols hide.

## Exercises (each is a second study, and each protocol is a fork of yours)

1. **The β interaction.** Crossing Lab 05 with this design: does the off/on-policy gap depend
   on the choice of divergence in the loss? (The literature says on-policy tolerates reverse
   KL better.)
2. **Scale one axis.** Re-run with the 1.7B teacher swapped for a served 8B-class teacher
   (Lab 08's topology). Which conclusions survive a 5× teacher?
3. **The sequencing curve.** Between pure off-policy and warm-started on-policy lies a
   schedule: {0%, 25%, 50%, 75%} of the budget spent off-policy first. Map the curve; find
   the knee, the point where more off-policy time stops helping; compare with Lab 07's
   cold-start findings.
4. **Publish the negative.** Whatever H2 returned, write the two-paragraph result note you
   would post publicly, including the seed ranges and the protocol hash. Negative results
   with auditable protocols are how a field learns; be the person who ships them.